# Record Reachy joint limits, one joint at a time
Only the chosen joint goes limp. Every other joint stays stiff and holds its pose, so you're moving a single axis and nothing else can flop around.

You move that joint by hand to the furthest point you trust in each direction, without ever going through the zone where the wires catch. The arc you covered, pulled inward by `MARGIN` degrees at each end, is saved as that joint's safe range in **`joint_limits.json`**. This works even when the arc crosses the point where the reported angle wraps (+180 → −180). Joints already in that file are kept. To do the next joint, change `JOINT` and run again from section 2.

Other code reads the limits through `joint_limits.py` (`check_limits`, `safe_goto`). Angles are in degrees.

**Posture matters:** some limits depend on where the other joints are (e.g. how far the elbow can bend before the hand hits the torso). Before freeing a joint, put the rest of the arm in a neutral pose away from the torso and table.

In [ ]:
PART = 'r_arm'               # 'r_arm', 'l_arm' or 'head'
JOINT = 'r_shoulder_pitch'   # the one joint to record
MARGIN = 3.0                 # degrees to pull each limit inward
HOST = None                  # None = findIP.get_reachy_ip(), or e.g. '169.254.50.2'

## 1. Connect

In [ ]:
import json, time
from datetime import datetime
from IPython.display import clear_output
from reachy_sdk import ReachySDK
from joint_limits import LIMITS_FILE

if HOST is None:
    from findIP import get_reachy_ip
    HOST = get_reachy_ip()
reachy = ReachySDK(host=HOST)

part = getattr(reachy, PART)
print('joints in', PART, ':', list(part.joints.keys()))

## 2. Pick the joint and free it
Set `JOINT` in the settings cell first if you're moving on to the next joint.

This cell stiffens the whole part where it is right now, then makes **only `JOINT`** limp. Hold the arm anyway: whatever hangs past the freed joint can swing under gravity.

In [ ]:
assert JOINT in part.joints, f'{JOINT} not in {PART}: {list(part.joints.keys())}'
joint = part.joints[JOINT]

reachy.turn_on(PART)       # every joint holds its current position
time.sleep(0.5)
joint.compliant = True     # free just this one
print(f'{JOINT} is limp at {joint.present_position:.1f} deg; everything else is stiff')

## 3. Record
Run the cell. From where the joint is now, move it slowly **one way** until it reaches the furthest point you trust, then back past the start and slowly **the other way** to its furthest safe point. **Never push it through the zone where the wires catch.** Press **Interrupt (■)** to stop.

`travel` is the total arc covered. `wraps` counts how often the reported angle jumped across its ±180 seam; that's fine and handled. If `travel` gets close to 360, you've gone all the way round and there's no forbidden zone left to protect.

In [ ]:
from joint_limits import intervals_from_samples, WRAP_JUMP_DEG

samples = [joint.present_position]
last_print = 0.0
try:
    while True:
        samples.append(joint.present_position)
        if time.time() - last_print > 0.3:
            last_print = time.time()
            safe, travel = intervals_from_samples(samples, margin=0)
            wraps = sum(abs(b - a) > WRAP_JUMP_DEG for a, b in zip(samples, samples[1:]))
            clear_output(wait=True)
            print(f'Recording {JOINT}... Interrupt (■) to stop.\n')
            print(f'now {samples[-1]:8.1f}   travel {travel:6.1f} deg   wraps {wraps}')
            print('covered:', '  '.join(f'[{lo:.1f}, {hi:.1f}]' for lo, hi in safe))
            if travel > 330:
                print('\n!! almost a full turn. Stop going this way.')
        time.sleep(0.02)
except KeyboardInterrupt:
    print('\nstopped.')

## 4. Review

In [ ]:
safe, travel = intervals_from_samples(samples, margin=MARGIN)
entry = None
print(f'travel {travel:.1f} deg')
if travel >= 360 - 2 * MARGIN:
    print('Went (almost) all the way round, so no forbidden zone was recorded. Not saving. Record again.')
elif not safe:
    print('Nothing left after the margin: did you move it?')
else:
    entry = {'safe': safe, 'travel': round(travel, 1), 'margin': MARGIN,
             'recorded': datetime.now().isoformat(timespec='seconds')}
    print(f'{JOINT} safe:', '  +  '.join(f'[{lo:.1f}, {hi:.1f}]' for lo, hi in safe))
    if len(safe) > 1:
        print('(the safe arc crosses the reported ±180 seam, so it is stored as two pieces; '
              'moves between the pieces will be refused, because goto would go the long way round)')
    if LIMITS_FILE.exists():
        old = json.loads(LIMITS_FILE.read_text()).get(JOINT)
        if old:
            print('currently saved:', old['safe'], f'(recorded {old["recorded"]})')

## 5. Save
Set `SAVE = True` once the numbers above look right. This overwrites any saved limit for this joint.

In [ ]:
SAVE = False

if SAVE and entry:
    data = json.loads(LIMITS_FILE.read_text()) if LIMITS_FILE.exists() else {}
    data[JOINT] = entry
    LIMITS_FILE.write_text(json.dumps(dict(sorted(data.items())), indent=2) + '\n')
    print(f'saved {JOINT} to {LIMITS_FILE}')
else:
    print('not saved')

## 6. Lock the joint again
Holds `JOINT` stiff where it is now. Then change `JOINT` in the settings cell and go back to section 2 for the next joint.

In [ ]:
joint.goal_position = joint.present_position   # so it doesn't snap to an old target
joint.compliant = False
print(f'{JOINT} stiff at {joint.present_position:.1f} deg')

## 7. Finished with all joints
Shows what's saved, then lets the part down gently. Hold the arm while it relaxes.

In [ ]:
if LIMITS_FILE.exists():
    for name, l in json.loads(LIMITS_FILE.read_text()).items():
        print(f'{name:22s}', '  +  '.join(f'[{lo:.1f}, {hi:.1f}]' for lo, hi in l['safe']))
reachy.turn_off_smoothly(PART)